In [1]:
# import gzip
# import importlib
# import math
# import struct
# import time
from dataclasses import dataclass, replace

import mnist

import numpy as np
# import plotly.graph_objects as go
# from plotly.subplots import make_subplots
from tqdm.auto import tqdm

import spikecorec as spc
from spikecorec import small_world_torus, square_torus, random_fixed_outdegree
from spikecorec import SpikeEngine

mnist.datasets_url = "https://storage.googleapis.com/cvdf-datasets/mnist/"
print(f"numpy {np.__version__}; spc {spc.__version__}")

numpy 2.2.6; spc 0.1.0


In [2]:
# get mnist data and load it 
def load_mnist():
    return (
        mnist.train_images(),
        mnist.train_labels(),
        mnist.test_images(),
        mnist.test_labels()
    )

train_images, train_labels, test_images, test_labels = load_mnist()

In [3]:
# initialize spiking neural network 
RANK = 1
MNIST_DATA_SCALE = 28
SPIKE_TAU = 10.0
INPUT_GAIN = 1.15
RECURRENT_SCALE = 1.03
VOLTAGE_SCALE = 1.0

pre_steps = 3
on_steps = 32
off_steps = 8

def initialize_network_for_mnist(reservoir_scale: int, seed: int, learning_rate = 0.00222, decay_rate = 0.22, resting_mp = 0.1) -> SpikeEngine:
    assert reservoir_scale > 0

    side_length = MNIST_DATA_SCALE * reservoir_scale
    network = small_world_torus(side_length, seed=seed)
    engine = SpikeEngine(
        network,
        [side_length, side_length],
        rank = RANK, 
        resting_mp = resting_mp, 
        decay_rate = decay_rate,
        learning_rate = learning_rate
    )

    # Map each of the 28x28 input pixels onto the center of its (scale x scale)
    # block on the side_length x side_length reservoir grid, row-major to match
    # the prepare_image_input flatten (row * 28 + col). Produces 784 distinct
    # input-neuron ids spread across the reservoir.
    scale = side_length // MNIST_DATA_SCALE      # block size (== reservoir_scale)
    offset = scale // 2                          # center of each block
    center_neuron_ids = [
        (scale * row_index + offset) * side_length + (scale * column_index + offset)
        for row_index in range(MNIST_DATA_SCALE)
        for column_index in range(MNIST_DATA_SCALE)
    ]

    engine.set_input_neurons(center_neuron_ids)

    # Tune the recurrent weights to sit near the bifurcation point (edge of
    # chaos, where the reservoir actually computes) and freeze plasticity so the
    # reservoir is FIXED during training — a given input always maps to the same
    # features. freeze_learning=True sets learning_rate=0. Without this the
    # reservoir is unconditioned and drifts every step (the reference relies on
    # the equivalent scale_*_near_bifurcation call).
    engine.scale_uniform_weights_near_bifurcation(scale=RECURRENT_SCALE, freeze_learning=True)
    return engine


In [4]:
# shared feature-extraction helpers — used by training, calibration, inference AND
# test so every code path sees identically-distributed reservoir features.

INPUT_NEURON_COUNT = MNIST_DATA_SCALE * MNIST_DATA_SCALE      # 784 input pixels
BACKGROUND_NOISE_INPUT = np.full((INPUT_NEURON_COUNT,), 0.1)
BLANK_INPUT = np.zeros((INPUT_NEURON_COUNT,))


def prepare_image_input(images, input_shape=(MNIST_DATA_SCALE, MNIST_DATA_SCALE)):
    rows, columns = input_shape
    X = images.astype(np.float32) / 255.0
    return np.ascontiguousarray(X.reshape(len(images), rows * columns), dtype=np.float32)


def run_image(engine: SpikeEngine, flat_image):
    """Drive the reservoir through ONE image from a clean state and return its
    reservoir feature vector. Schedule: pre_steps background noise -> on_steps
    image (scaled by INPUT_GAIN) -> off_steps blank settling, feature read at the
    final tick.

    This is THE single feature-extraction path. Per-image local ticks (0..N-1)
    are used: only tick *differences* matter for the spike traces, and because
    every call (train / calibrate / infer / test) runs this identical schedule,
    the feature distribution is the same everywhere and the probe's normalizer
    absorbs any constant offset. (Previously training used a monotonic-tick
    windowed loop while inference skipped the off-steps — a train/test mismatch.)
    """
    engine.reset_state(0)
    driven = flat_image * np.float32(INPUT_GAIN)     # input gain on the image drive
    tick = 0
    for _ in range(pre_steps):
        engine.step_simulation(BACKGROUND_NOISE_INPUT, tick=tick); tick += 1
    for _ in range(on_steps):
        engine.step_simulation(driven, tick=tick); tick += 1
    for _ in range(off_steps):
        engine.step_simulation(BLANK_INPUT, tick=tick); tick += 1
    return engine.get_reservoir_features_vector(tick - 1, SPIKE_TAU, VOLTAGE_SCALE)


def collect_features(engine: SpikeEngine, images, n=256, seed=0):
    """Run `n` randomly-chosen images through the reservoir and stack their
    feature vectors — for calibrating a probe's normalizer on the REAL feature
    distribution (instead of random gaussian noise)."""
    rng = np.random.default_rng(seed)
    n = min(int(n), len(images))
    idx = rng.choice(len(images), size=n, replace=False)
    flats = prepare_image_input(images[idx])
    feats = np.empty((n, 2 * engine.neuron_count + 1), dtype=np.float32)
    for i in range(n):
        feats[i] = run_image(engine, flats[i])
    return feats


In [5]:
# initialize network and probe layer 
from rls_probe import OnlineRLSProbe
from softmax_probe import OnlineSoftmaxProbe

engine = initialize_network_for_mnist(reservoir_scale = 3, seed=1042)

number_of_features = engine.neuron_count * 2 + 1
probe = OnlineRLSProbe(number_of_features)
# Lower lr + stronger L2 than the defaults: 14113 features on single-pass online
# SGD overfits hard (confident-but-wrong, huge test CE), so regularize more.
probe_softmax = OnlineSoftmaxProbe(number_of_features, lr=1e-2, l2=1e-3)

# Calibrate the feature normalizer on REAL reservoir features (warmup images run
# through the same run_image() schedule used for training), not random gaussian
# noise. Both probes share the same calibration set.
calibration_features = collect_features(engine, train_images, n=256, seed=0)
probe.set_normalizer(calibration_features)
probe_softmax.set_normalizer(calibration_features)


In [6]:
# training loop — features come from the shared run_image() schedule, so the
# probe trains on exactly the features it will see at calibration/inference/test.

def train_network(engine: SpikeEngine, train_data: np.ndarray, train_labels: np.ndarray,
                  probe, **fit_kwargs):
    flats = prepare_image_input(train_data)
    loss_per_example = []
    accuracy_per_example = []
    for i in tqdm(range(len(train_labels)), total=len(train_labels)):
        features = run_image(engine, flats[i])
        loss, accuracy = probe.partial_fit(features, train_labels[i], **fit_kwargs)
        loss_per_example.append(loss)
        accuracy_per_example.append(accuracy)
    return loss_per_example, accuracy_per_example
#
# # RLS probe takes a forgetting factor; the softmax probe (trained below) does not.
# train_network(engine, train_images, train_labels, probe, forgetting=0.995)


In [7]:
# inference — uses the same run_image() schedule as training/test

def inference(engine: SpikeEngine, sample: np.ndarray, probe):
    flat = prepare_image_input(sample[None, ...],
                               input_shape=(sample.shape[0], sample.shape[1]))[0]
    features = run_image(engine, flat)
    return probe.predict(features)


In [8]:
# train the softmax probe (train_network is defined above; run_image gives the
# same features the probe will see at inference/test time).
# Note: this trains on the *current* engine. To train the softmax probe on a
# fresh reservoir, re-run the init cell above first.
train_network(engine, train_images, train_labels, probe_softmax)


  0%|          | 0/60000 [00:00<?, ?it/s]

([2.3025848865509033,
  2.357304811477661,
  2.2071337699890137,
  2.222578763961792,
  2.1854612827301025,
  2.2356765270233154,
  2.1891086101531982,
  2.655893564224243,
  0.8108648061752319,
  1.7528775930404663,
  1.5339707136154175,
  3.19950270652771,
  4.58309268951416,
  2.386605739593506,
  0.4650474786758423,
  2.5128185749053955,
  1.705339789390564,
  2.701355218887329,
  1.7766603231430054,
  2.69391131401062,
  18.42068099975586,
  0.2755582928657532,
  1.324820637702942,
  0.7262126207351685,
  2.1227424144744873,
  1.6631617546081543,
  2.478107213973999,
  0.019060086458921432,
  18.42068099975586,
  2.6725080013275146,
  6.9052348136901855,
  1.625020146369934,
  0.4143997132778168,
  0.775595486164093,
  1.5621987581253052,
  1.6958340406417847,
  1.1705034971237183,
  0.07422072440385818,
  2.5402402877807617,
  0.22632664442062378,
  0.6332874894142151,
  1.4222239255905151,
  2.209249258041382,
  0.5860198736190796,
  0.9439300894737244,
  0.8147401809692383,
  2

In [9]:
def test_network(engine: SpikeEngine, probe, test_data: np.ndarray, test_labels: np.ndarray,
                 max_samples: int | None = None, n_classes: int = 10):
    """Evaluate `probe` on the test set using the shared run_image() schedule
    (the same features as training). Returns a metrics dict and prints accuracy,
    per-digit accuracy, and a confusion matrix."""
    flats = prepare_image_input(test_data)
    targets = np.asarray(test_labels, dtype=np.int64)
    n = len(targets) if max_samples is None else min(int(max_samples), len(targets))

    preds = np.empty(n, dtype=np.int64)
    losses = np.empty(n, dtype=np.float64)
    for i in tqdm(range(n), desc="evaluating"):
        features = run_image(engine, flats[i])
        probs = probe.probabilities(features)[0]            # (n_classes,)
        label = int(targets[i])
        preds[i] = int(probs.argmax())
        losses[i] = float(-np.log(probs[label] + 1e-8))

    # ---- metrics ----
    y = targets[:n]
    correct = preds == y
    accuracy = float(correct.mean())
    mean_loss = float(losses.mean())
    confusion = np.zeros((n_classes, n_classes), dtype=np.int64)
    for t, p in zip(y, preds):
        confusion[t, p] += 1
    class_totals = confusion.sum(axis=1)
    per_class_accuracy = np.divide(np.diag(confusion), class_totals,
                              out=np.zeros(n_classes), where=class_totals > 0)

    # ---- summary ----
    print(f"\n{probe.readout_name} on {n} test examples")
    print(f"  accuracy    : {accuracy * 100:.2f}%  ({int(correct.sum())}/{n})")
    print(f"  mean CE loss: {mean_loss:.4f}")
    print("\n  per-digit accuracy:")
    for c in range(n_classes):
        bar = "#" * int(round(per_class_accuracy[c] * 20))
        print(f"    {c}: {per_class_accuracy[c] * 100:5.1f}%  ({int(confusion[c, c])}/{int(class_totals[c])})  {bar}")
    print("\n  confusion matrix (rows = true, cols = predicted):")
    print("        " + " ".join(f"{c:>4}" for c in range(n_classes)))
    for t in range(n_classes):
        row = " ".join(f"{confusion[t, p]:>4}" for p in range(n_classes))
        print(f"    {t} | {row}")

    return {
        "accuracy": accuracy,
        "mean_loss": mean_loss,
        "per_class_accuracy": per_class_accuracy,
        "confusion_matrix": confusion,
        "predictions": preds,
        "labels": y,
    }

result = test_network(engine, probe_softmax, test_images, test_labels)


evaluating:   0%|          | 0/10000 [00:00<?, ?it/s]


online softmax probe (fast) on 10000 test examples
  accuracy    : 87.25%  (8725/10000)
  mean CE loss: 1.4275

  per-digit accuracy:
    0:  94.9%  (930/980)  ###################
    1:  97.5%  (1107/1135)  ####################
    2:  81.5%  (841/1032)  ################
    3:  86.3%  (872/1010)  #################
    4:  86.0%  (845/982)  #################
    5:  83.2%  (742/892)  #################
    6:  90.6%  (868/958)  ##################
    7:  87.9%  (904/1028)  ##################
    8:  76.0%  (740/974)  ###############
    9:  86.8%  (876/1009)  #################

  confusion matrix (rows = true, cols = predicted):
           0    1    2    3    4    5    6    7    8    9
    0 |  930    0    6    5    3   11   18    4    3    0
    1 |    0 1107    6    3    3    1    3    2   10    0
    2 |   11   13  841   54   20    6   42   12   33    0
    3 |    3    1   26  872    3   74    3   10   15    3
    4 |    1    4    6   13  845   21   21   14    4   53
    5 |   11  

In [10]:
# compare: the RLS probe on the same test set. RLS is regularized (delta) and
# far less prone to the single-pass overfitting the softmax probe showed, so
# it's a useful baseline. The reservoir is frozen, so testing it here is safe.
rls_result = test_network(engine, probe, test_images, test_labels)


evaluating:   0%|          | 0/10000 [00:00<?, ?it/s]

KeyboardInterrupt: 